### Setup

In [17]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import json
import torch
from torchvision import transforms
from PIL import Image
from torchmetrics.image.fid import FrechetInceptionDistance

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
TEST_SIZE = 5

# PATH
## Augment Instruction Performace Data
SEED_IMAGE_FOLDER = '../Data/Seed/Seed_Image'
SEED_IMAGE_FILE = sorted(os.listdir(SEED_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SEED_LABEL_FOLDER ='../Data/Seed/EN_Seed_Label'
SDDE_LABEL_FILE = sorted(os.listdir(SEED_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_IMAGE_FOLDER = '../Data/Augment/Augment_KFashion_Image'
AUGMENT_IMAGE_FILE = sorted(os.listdir(AUGEMNT_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_LABEL_FOLDER =  '../Data/Augment/Augment_CLIP'
AUGMENT_LABEL_FILE = sorted(os.listdir(AUGEMNT_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## Generate of Model Performance-Prompt Following Performance
PROMPT_FOLDER = '../Data/Generate/Prompt'
PROMPT_FILE = sorted(os.listdir(PROMPT_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = '../Data/Generate/Generate_Image/StableDiffusion_Image'
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = '../Data/Generate/Generate_Image/FIGMA_Image'
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## Generate of Model Performance-Image Following Performance
ANSWER_IMAGE_FOLDER = '../Data/Generate/Answer_Image'
ANSWER_IMAGE_FILE = sorted(os.listdir(ANSWER_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = '../Data/Generate/Generate_Image/StableDiffusion_Image'
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = '../Data/Generate/Generate_Image/FIGMA_Image'
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250226_130326'

device :  cuda


### Augment Instruction Performace

- seed image - seed label 

In [55]:
from metric import calculate_clip_score 
prompt_folder = SEED_LABEL_FOLDER
image_folder = SEED_IMAGE_FOLDER

clip_scores = []
for filename in os.listdir(prompt_folder)[:TEST_SIZE]:
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        # 딕셔너리 내의 각 항목에서 필요한 문자열 추출
        persona = prompt_data.get("Prompt", {}).get("persona", "")
        task_description = prompt_data.get("Prompt", {}).get("task_description", "")
        constraint = prompt_data.get("Prompt", {}).get("constraint", "")
        caption = prompt_data.get("Input", {}).get("caption", "")
        add_info = prompt_data.get("Add_Info", "")
        
        # 모든 항목을 하나의 문자열로 결합
        final_prompt = " ".join([persona, task_description, constraint, caption, add_info])
    
        final_prompt = final_prompt[:77]  # CLIP 모델의 최대 토큰 수에 맞춤
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        print(f"File: {filename}, CLIP Score: {clip_score:.4f}")
        
if clip_scores:
    average_clip_score = sum(clip_scores) / len(clip_scores)
    print(f"Average CLIP Score: {average_clip_score:.4f}")
else:
    print("No valid CLIP scores calculated")

Average CLIP Score: 0.2514


- augment image - augment label

### Generate Model Performance-Prompt Following Performance

- stablediffusion image - prompt

- figma image - prompt

### Generate Model Performance-Image Following Performance

In [1]:
def calculate_fid(answer_image_folder, sd_image_folder):
    # 폴더 내 이미지 파일 목록 가져오기
    answer_image_files = sorted(os.listdir(answer_image_folder), key=lambda x: int(x.split('.')[0]))
    sd_image_files = sorted(os.listdir(sd_image_folder), key=lambda x: int(x.split('.')[0]))

    # 두 폴더의 이미지 수가 같은지 확인
    assert len(answer_image_files) == len(sd_image_files), "두 폴더의 이미지 수가 같아야 합니다."

    # 디바이스 설정 (GPU 사용 가능 시 GPU 사용)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # FID 객체 생성
    fid = FrechetInceptionDistance().to(device)

    # 이미지 변환 설정
    transform = transforms.Compose([
        transforms.Resize((299, 299)),  # InceptionV3 입력 크기
        transforms.ToTensor(),          # 텐서로 변환
        transforms.Lambda(lambda x: x * 255),  # [0, 1] 범위를 [0, 255]로 변환
        transforms.Lambda(lambda x: x.byte())  # torch.uint8 타입으로 변환
    ])

    # 정답 이미지 추가
    for img_name in answer_image_files:
        img_path = os.path.join(answer_image_folder, img_name)
        img = Image.open(img_path).convert("RGB")
        img = transform(img).unsqueeze(0).to(device)  # 배치 차원 추가
        fid.update(img, real=True)

    # 생성된 이미지 추가
    for img_name in sd_image_files:
        img_path = os.path.join(sd_image_folder, img_name)
        img = Image.open(img_path).convert("RGB")
        img = transform(img).unsqueeze(0).to(device)  # 배치 차원 추가
        fid.update(img, real=False)

    # FID 계산
    fid_score = fid.compute().item()
    return fid_score

In [15]:
avg_fid_score = calculate_fid(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)
print(f"Average FID Score: {avg_fid_score:.4f}")

Average FID Score: 167.6022


- stablediffusion image - answer image

- figma image - answer image